# 텍스트 벡터화 복습

이 노트북은 `03_text_vectorization`의 전체 흐름(BoW·TF-IDF·N-gram, Word2Vec·GloVe·FastText, PyTorch `nn.Embedding`, BERT·Doc2Vec, LSA·LDA)을 문제로 복습합니다. 총 60문제를 순서대로 풀며 각 빈 코드 셀에 직접 풀이해 보세요.

- 실행 환경에 따라 `gensim`, `scikit-learn`, `torch`가 필요합니다.
- 작은 예제의 유사도 값은 난수와 학습 환경에 따라 조금 달라질 수 있으므로, **값 자체보다 비교 방향과 shape**을 확인하세요.

## 원본 노트북과 문제 연결

- `01_text_vectorization.ipynb`: 문제 10, 16, 19~27, 30, 45
- `02_bow_tfidf.ipynb`: 문제 1~4, 13, 25, 28~30
- `03_word_embedding.ipynb`: 문제 5, 14~18, 31~35
- `04_torch_Embedding.ipynb`: 문제 6~8, 36~40, 46~60
- `05_fasttext.ipynb`: 문제 9, 41~44

일부 문제는 두 개 이상의 노트북 개념을 함께 사용하므로, 위 번호는 가장 직접적인 원본 단원을 기준으로 연결했습니다.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from gensim.models import Word2Vec, FastText


## 문제 1. BoW와 DTM 읽기

다음 네 문서를 `CountVectorizer`로 문서-단어 행렬(DTM)로 바꾸세요.

```python
documents = [
    "I love my dog",
    "I love my cat",
    "my dog loves a walk",
    "a cat sleeps",
]
```

조건:

- `CountVectorizer()`를 만들고 `fit_transform()` 결과를 `bow_matrix`에 저장하세요.
- 학습된 어휘를 열 이름으로 하는 DataFrame `dtm_df`를 만드세요. 행 이름은 `doc1`~`doc4`로 지정합니다.
- DTM의 shape과 `doc3`에서 `dog`, `walk`의 빈도를 출력하세요.
- 단어 순서가 바뀐 `"dog my love I"`의 BoW 벡터가 `doc1`과 같은지 확인하세요.

확인할 점:

- DTM의 행은 문서, 열은 어휘입니다.
- BoW는 단어의 **등장 횟수**를 남기고 단어 **순서**는 잃습니다.

힌트: `vectorizer.get_feature_names_out()`, `bow_matrix.toarray()`

## 문제 2. DTM과 TDM의 축 바꾸기

문제 1의 `dtm_df`를 전치해 단어-문서 행렬(TDM)을 만드세요.

조건:

- `dtm_df.T`로 `tdm_df`를 만드세요.
- DTM과 TDM의 shape을 각각 출력하세요.
- `dtm_df.loc["doc3", "dog"]`와 `tdm_df.loc["dog", "doc3"]`가 같은지 확인하세요.
- `dog` 행을 출력해 어떤 문서에 이 단어가 등장하는지 확인하세요.
- 전치가 빈도 값을 바꾸는 연산인지, 행과 열을 읽는 관점만 바꾸는 연산인지 주석으로 작성하세요.

힌트: `DataFrame.T`

## 문제 3. TF-IDF와 문서 유사도

문제 1의 `documents`로 TF-IDF를 만들고 BoW와 유사도를 비교하세요.

조건:

- `TfidfVectorizer()`의 결과를 `tfidf_matrix`, DataFrame을 `tfidf_df`에 저장하세요.
- `cat`, `dog`, `my`의 IDF를 출력하고, 여러 문서에 등장하는 `my`의 가중치가 왜 낮아질 수 있는지 주석으로 설명하세요.
- BoW와 TF-IDF 각각에 대해 코사인 유사도 행렬을 계산하세요.
- `doc1-doc2`, `doc1-doc3`의 유사도를 두 방식에서 비교하세요.
- TF-IDF 벡터의 각 행 L2 norm이 약 1인지 확인하세요.

힌트:

```python
tfidf_vectorizer.idf_
cosine_similarity(matrix)
np.linalg.norm(tfidf_matrix.toarray(), axis=1)
```

## 문제 4. 단어 N-gram으로 짧은 순서 정보 추가하기

다음 두 문장은 단어 구성은 같지만 의미가 반대가 될 수 있습니다. unigram과 bigram 표현을 비교하세요.

```python
sentiment_docs = [
    "this movie is not good",
    "this movie is good",
    "this movie is not bad",
]
```

조건:

- `ngram_range=(1, 1)`과 `(1, 2)`로 각각 CountVectorizer를 학습하세요.
- 두 어휘 목록과 shape을 비교하세요.
- bigram 어휘에 `not good`, `not bad`가 포함되는지 확인하세요.
- unigram과 unigram+bigram 각각에서 문서 1과 문서 2의 코사인 유사도를 구하세요.
- N-gram이 긴 문맥을 완전히 이해하지는 못하는 이유와, N을 크게 하면 생길 수 있는 비용을 주석으로 작성하세요.

힌트: `CountVectorizer(ngram_range=(1, 2))`

## 문제 5. Skip-gram Word2Vec 학습과 OOV

문장별 토큰 목록으로 작은 Skip-gram Word2Vec을 학습하고, 임베딩 행렬을 읽으세요.

```python
base_sentences = [
    ["king", "is", "a", "royal", "man"],
    ["queen", "is", "a", "royal", "woman"],
    ["prince", "is", "a", "young", "royal"],
    ["apple", "is", "a", "sweet", "fruit"],
]
sentences = base_sentences * 200
```

조건:

- `vector_size=20`, `window=2`, `min_count=1`, `sg=1`, `epochs=100`, `seed=42`, `workers=1`로 모델을 학습하세요.
- `key_to_index`와 `vectors.shape`를 출력하고, `king` 벡터의 shape을 확인하세요.
- `king`과 `queen`, `king`과 `apple`의 코사인 유사도를 비교하세요.
- `kings`가 학습 어휘에 있는지 확인하고, 일반 Word2Vec에서 OOV 벡터 조회가 실패하는 이유를 주석으로 설명하세요.
- CBOW와 Skip-gram의 입력·예측 방향 차이를 한 줄씩 주석으로 작성하세요.

힌트: `Word2Vec(sentences=..., sg=1, ...)`, `model.wv.similarity()`

## 문제 6. `nn.Embedding` 조회와 PAD 마스킹

아래 토큰 ID 배치에서 0은 `<PAD>`입니다. `nn.Embedding`으로 벡터를 조회하고 PAD를 제외한 문장 평균을 구하세요.

```python
word_to_id = {"<PAD>": 0, "<OOV>": 1, "good": 2, "movie": 3, "not": 4, "bad": 5}
input_ids = torch.tensor([
    [2, 3, 0, 0],
    [4, 2, 3, 0],
    [5, 3, 0, 0],
])
```

조건:

- `num_embeddings=len(word_to_id)`, `embedding_dim=4`, `padding_idx=0`으로 임베딩 층을 만드세요.
- 출력 `embedded`의 shape과 `embedding.weight[2]`와 `embedded[0, 0]`의 일치 여부를 출력하세요.
- PAD 행이 0 벡터인지 확인하세요.
- `(input_ids != 0)` 마스크로 PAD를 분자와 분모에서 모두 제외한 `sentence_vectors`를 만드세요.
- 단순 평균과 마스크 평균이 첫 번째 문장에서 왜 달라질 수 있는지 주석으로 작성하세요.

힌트:

```python
mask = (input_ids != 0).unsqueeze(-1)
masked_sum = (embedded * mask).sum(dim=1)
lengths = mask.sum(dim=1).clamp(min=1)
```

## 문제 7. 평균 임베딩 분류기의 입출력 계약

문제 6의 `sentence_vectors`를 입력으로 받아 긍정(1)·부정(0) 두 클래스를 분류하는 선형층을 만드세요.

```python
labels = torch.tensor([1, 1, 0])
```

조건:

- `nn.Linear(4, 2)`로 분류기를 만들고 로짓의 shape을 출력하세요.
- `nn.CrossEntropyLoss()`로 손실을 계산하세요. 모델 출력에 softmax를 먼저 적용하지 마세요.
- Adam optimizer로 50회 이상 학습하며 손실이 어떻게 변하는지 출력하세요.
- `argmax(dim=1)`로 예측 클래스와 학습 데이터 정확도를 출력하세요.
- 평균 임베딩이 `good movie`와 `movie good`을 구분하지 못하는 이유를 주석으로 작성하세요.

힌트: 학습 순서는 `zero_grad() → logits 계산 → loss 계산 → backward() → step()`입니다.

## 문제 8. 사전학습 임베딩 행 정렬과 동결

사전학습 벡터는 현재 작업의 `word_to_id` 순서에 맞춰 행렬을 만들어야 합니다. 아래 예제 벡터를 이용해 초기화 과정을 구현하세요.

```python
pretrained = {
    "good": np.array([0.8, 0.1, 0.3, 0.5], dtype=np.float32),
    "movie": np.array([0.2, 0.4, 0.7, 0.1], dtype=np.float32),
    "bad": np.array([-0.7, 0.2, -0.4, 0.3], dtype=np.float32),
}
```

조건:

- `(len(word_to_id), 4)` 크기의 0 행렬을 만들고, 사전에 있는 단어의 벡터만 해당 ID 행에 복사하세요.
- `<PAD>`, `<OOV>`, `not` 행이 어떤 값인지 출력하세요.
- `nn.Embedding.from_pretrained()`로 `freeze=True`, `freeze=False` 임베딩을 각각 만드세요.
- 두 weight의 `requires_grad`와 `movie` 행이 원본 벡터와 같은지 확인하세요.
- 어휘 ID와 사전학습 행 순서가 어긋나도 shape 오류 없이 잘못된 단어 벡터를 조회할 수 있는 이유를 주석으로 설명하세요.

힌트: `torch.tensor(embedding_matrix)`, `freeze`

## 문제 9. FastText 문자 n-gram과 OOV

FastText가 `apples`처럼 학습 어휘에 없는 단어에도 벡터를 만들 수 있는 과정을 확인하세요.

조건:

- `<apple>`, `<apples>`에서 문자 3-gram을 만드는 함수 `char_ngrams(word, n=3)`를 작성하세요.
- 두 단어의 n-gram 집합과 공통 n-gram을 출력하세요.
- 문제 5의 `sentences`로 Word2Vec과 FastText를 동일하게 학습하되 FastText에는 `min_n=3`, `max_n=6`, `bucket=10_000`을 지정하세요.
- `apples`가 두 모델의 `key_to_index`에 있는지 확인하세요.
- Word2Vec의 `get_vector("apples")` 예외를 처리하고, FastText의 같은 벡터 shape을 출력하세요.
- `apple-apples`, `apple-queen` FastText 유사도를 비교하고, 철자가 비슷하다고 의미까지 정확히 비슷한 것은 아닌 이유를 주석으로 작성하세요.

힌트: `f"<{word}>"`, `FastText(sentences=..., min_n=3, max_n=6, ...)`

## 문제 10. 벡터화 방법 선택하기

아래 상황마다 가장 먼저 시도할 벡터화 방법을 하나 고르고, 이유와 한계를 2~3문장으로 작성하세요. 정답 하나만 있는 문제가 아니라 **표현 대상·보존할 정보·OOV·해석 가능성·비용**을 근거로 선택하는 문제입니다.

1. 고객 문의 5만 건을 빠르게 분류해야 하고, 어떤 단어가 분류에 영향을 주었는지 설명해야 한다.
2. 형태 변화와 신조어가 많은 상품 검색어에서 처음 보는 단어도 처리해야 한다.
3. 작은 감성 데이터셋이 있고, 이미 비슷한 도메인에서 학습된 단어 벡터를 활용하고 싶다.
4. `bank`처럼 같은 토큰이 문장 문맥에 따라 다른 의미를 가져야 한다.
5. 수천 개 문서의 잠재 주제 비율을 사람이 읽고 해석해야 한다.

힌트:

- BoW·TF-IDF: 빠른 기준선, 해석 가능, 단어 순서·문맥 한계
- Word2Vec: 정적 단어 의미, 고정 어휘 OOV 한계
- FastText: 문자 n-gram으로 OOV 완화, 철자 유사성의 한계
- 사전학습 임베딩: 작은 데이터의 좋은 시작점, 언어·도메인·토큰화 정렬 필요
- Transformer 계열: 문맥적 표현, 더 큰 계산 비용
- LDA: 문서별 주제 비율 해석, 단어 순서 한계

## 문제 11. 정수 인코딩·원-핫 인코딩

다음 토큰 문장을 정수 ID와 원-핫 벡터로 바꾸세요.

```python
tokenized = [["i", "love", "nlp"], ["i", "study", "nlp"]]
new_tokens = ["i", "enjoy", "nlp"]
```

- 빈도 내림차순·동률 알파벳순으로 어휘를 만들고 `<PAD>=0`, `<OOV>=1`을 배정하세요.
- 학습 문장과 `new_tokens`를 ID로 바꾸세요. `enjoy`는 OOV여야 합니다.
- `torch.nn.functional.one_hot`으로 원-핫 텐서를 만들고 shape을 출력하세요.
- ID 숫자 차이가 단어 의미의 거리나 순서를 뜻하지 않는 이유를 주석으로 작성하세요.

힌트: `torch.nn.functional.one_hot(torch.tensor(ids), num_classes=vocab_size)`

## 문제 12. 희소 벡터와 밀집 벡터의 차이

문제 1의 `bow_matrix`와 문제 5의 `model.wv.vectors`를 비교하세요.

- 각 행렬의 shape과 0의 비율을 계산하세요. 희소 행렬은 `toarray()`로 바꾼 뒤 계산합니다.
- BoW가 희소해지는 이유와 Word2Vec 벡터가 보통 밀집하는 이유를 주석으로 설명하세요.
- 어휘가 100,000개인 BoW에서 문서 하나를 `float32` 밀집 배열로 저장할 때 필요한 대략적인 바이트 수를 계산하세요.
- 이 경우 희소 행렬 저장 방식이 유리한 이유를 한 문장으로 작성하세요.

힌트: `(array == 0).mean()`, `100_000 * 4`

## 문제 13. 코사인 유사도를 직접 계산하기

다음 벡터로 코사인 유사도의 성질을 직접 확인하세요.

```python
x = np.array([1, 2, 0], dtype=float)
y = np.array([2, 4, 0], dtype=float)
z = np.array([0, 0, 3], dtype=float)
```

- 내적과 L2 norm을 사용해 `cos(x, y)`, `cos(x, z)`를 직접 계산하세요.
- `cosine_similarity()` 결과와 일치하는지 확인하세요.
- `y`가 `x`의 2배인데도 유사도가 1인 이유를 설명하세요.
- 영벡터와 코사인 유사도를 계산할 때 주의할 점을 주석으로 작성하세요.

힌트: `(x @ y) / (np.linalg.norm(x) * np.linalg.norm(y))`

## 문제 14. CBOW와 Skip-gram 학습 쌍 만들기

문장 `tokens = ["the", "royal", "queen", "smiles", "today"]`에서 중심 단어를 `queen`, window를 2로 두고 학습 관계를 작성하세요.

- `queen`의 주변 단어 목록을 구하세요.
- Skip-gram의 `(입력 중심 단어, 예측 주변 단어)` 쌍을 리스트로 만드세요.
- CBOW의 `(입력 주변 단어들, 예측 중심 단어)` 쌍을 만드세요.
- 두 방식 중 이 예제에서 학습 쌍이 더 많이 생성되는 방식을 확인하세요.
- `window=2`가 문장의 앞뒤 두 문장을 뜻하지 않는 이유를 주석으로 작성하세요.

힌트: 문맥 창은 중심 토큰의 앞뒤 **토큰 위치** 범위입니다.

## 문제 15. Word2Vec 설정값 바꿔 보기

문제 5의 코퍼스로 아래 두 Word2Vec 모델을 학습해 설정의 의미를 비교하세요.

- 모델 A: `sg=0`, `window=1` (CBOW)
- 모델 B: `sg=1`, `window=3` (Skip-gram)

조건:

- 두 모델의 어휘 수와 임베딩 행렬 shape을 출력하세요.
- `king-queen` 유사도와 `king`의 가까운 단어 3개를 각각 출력하세요.
- `sg`와 `window`가 각각 무엇을 바꾸는지 주석으로 작성하세요.
- 작은 반복 코퍼스에서 수치 차이를 일반적인 모델 우열로 판단할 수 없는 이유를 작성하세요.

힌트: 나머지 설정과 `seed`는 같게 유지하세요.

## 문제 16. GloVe·Word2Vec·FastText 비교표

아래 세 모델을 표로 비교하세요. 코드는 작성하지 않고 Markdown 또는 주석으로 답합니다.

| 비교 기준 | Word2Vec | GloVe | FastText |
| --- | --- | --- | --- |
| 학습 신호 |  |  |  |
| 기본 단위 |  |  |  |
| OOV 처리 |  |  |  |
| 강점 |  |  |  |
| 주의점 |  |  |  |

추가로, ‘Word2Vec과 GloVe는 정적 임베딩’이라는 말의 뜻과 FastText가 OOV 벡터를 만든다고 해서 새 단어의 실제 의미를 이해한 것은 아닌 이유를 각각 한 문장으로 정리하세요.

## 문제 17. PCA로 임베딩 2차원 시각화

문제 5 모델에서 `king`, `queen`, `prince`, `man`, `woman`, `apple`, `fruit`의 벡터를 골라 2차원으로 시각화하세요.

- `sklearn.decomposition.PCA(n_components=2)`를 사용해 2차원 좌표를 만드세요.
- matplotlib 산점도에 각 단어 라벨을 표시하세요.
- `explained_variance_ratio_`와 두 축의 합을 출력하세요.
- 2차원에서 가까운 단어가 원래 고차원 공간에서도 반드시 같은 순위로 가깝다고 단정할 수 없는 이유를 주석으로 작성하세요.

힌트: `vectors = np.vstack([model.wv[word] for word in words])`

## 문제 18. 모델 저장 방식 구분하기

문제 5의 Word2Vec 모델을 전체 모델과 벡터 전용 파일로 각각 저장·복원하는 코드를 작성하세요.

- `model.save()`와 `Word2Vec.load()`를 사용해 전체 모델을 저장·복원하세요.
- `model.wv.save_word2vec_format()`으로 벡터 전용 텍스트 파일을 저장하세요.
- 전체 모델 복원 후 `king` 벡터가 원본과 같은지 확인하세요.
- 전체 모델 저장과 벡터 전용 저장이 각각 어떤 경우에 알맞은지 주석으로 작성하세요.
- FastText에서 OOV 생성 기능을 보존하려면 어떤 저장 방식이 필요한지 문제 9의 결과와 연결해 설명하세요.

힌트: `np.allclose(original.wv["king"], restored.wv["king"])`

## 문제 19. BERT의 문맥적 표현 이해하기

다음 두 문장에서 `bank`가 어떤 의미인지 먼저 쓰고, 정적 임베딩과 문맥적 임베딩이 이 차이를 어떻게 다루는지 비교하세요.

```text
A. I deposited money at the bank.
B. We sat on the bank of the river.
```

- Word2Vec·GloVe·FastText가 기본적으로 `bank`에 몇 개의 벡터를 주는지 작성하세요.
- BERT 계열 모델이 두 문장의 `bank` 토큰에 다른 표현을 만들 수 있는 이유를 설명하세요.
- BERT 입력에서 subword 토큰화와 attention mask가 필요한 이유를 간단히 작성하세요.
- 문맥적 표현의 장점 하나와 계산 비용 또는 운영상 주의점 하나를 적으세요.


## 문제 20. Doc2Vec 입력과 문서 벡터

다음 세 문서를 Doc2Vec 학습 입력으로 준비하는 코드를 작성하세요.

```python
review_tokens = [
    ["movie", "was", "excellent"],
    ["movie", "was", "boring"],
    ["excellent", "acting", "and", "music"],
]
```

- `gensim.models.doc2vec.TaggedDocument`로 각 문서에 고유 tag를 붙이세요.
- `Doc2Vec(vector_size=20, min_count=1, epochs=100, seed=42)` 모델을 학습하세요.
- 저장된 문서 벡터 하나의 shape과 새 문서 `['excellent', 'movie']`의 `infer_vector()` shape을 출력하세요.
- Word2Vec이 주로 무엇을 벡터 하나로 만드는지, Doc2Vec이 추가로 무엇을 만드는지 주석으로 비교하세요.

힌트: `TaggedDocument(words=words, tags=[f"doc_{i}"])`

## 문제 21. LSA: 문서-단어 행렬 저차원화

문제 1의 BoW 행렬을 `TruncatedSVD`로 2차원 잠재 공간에 압축하세요.

- `TruncatedSVD(n_components=2, random_state=42)`를 학습하고 문서 좌표 `lsa_docs`를 만드세요.
- 원래 DTM과 `lsa_docs`의 shape을 비교하세요.
- `components_`의 각 잠재 축에서 절댓값이 큰 단어 3개를 출력하세요.
- `explained_variance_ratio_` 합을 출력하세요.
- LSA 축의 값이 음수를 포함할 수 있으므로 확률·주제 비율로 바로 해석할 수 없는 이유를 주석으로 작성하세요.

힌트: `np.argsort(np.abs(svd.components_[i]))[-3:]`

## 문제 22. LDA: 문서별 주제 비율 읽기

다음 코퍼스를 Count 행렬로 바꾼 뒤 주제 2개 LDA를 학습하세요.

```python
topic_docs = [
    "team scored a goal in the match",
    "player won the football match",
    "stock market price increased today",
    "investor bought shares in the market",
]
```

- `CountVectorizer`로 입력 행렬을 만드세요. TF-IDF가 아니라 빈도 행렬을 쓰는 이유도 주석으로 작성하세요.
- `LatentDirichletAllocation(n_components=2, random_state=42)`을 학습하세요.
- 각 주제의 상위 단어 5개와 각 문서의 주제 비율을 출력하세요.
- 문서별 주제 비율 행의 합이 약 1인지 확인하세요.
- LDA와 분류에서의 Linear Discriminant Analysis가 약어만 같고 다른 모델인 이유를 작성하세요.


## 문제 23. LSA와 LDA 비교

문제 21, 22의 결과를 바탕으로 다음 항목을 비교표로 정리하세요.

| 비교 기준 | LSA | LDA |
| --- | --- | --- |
| 입력 행렬 |  |  |
| 핵심 계산 |  |  |
| 문서 표현 |  |  |
| 값의 범위·해석 |  |  |
| 알맞은 목표 |  |  |

추가로, ‘잠재 축’과 ‘사람이 붙이는 주제 이름’이 자동으로 완전히 같은 것은 아닌 이유를 두 모델 각각에 대해 작성하세요.

## 문제 24. 방법 선택 근거를 수치로 확인하기

다음 가상의 서비스 로그를 보고, 벡터화 기법을 고르기 전에 확인할 지표와 검증 설계를 작성하세요.

```text
- 검색어 100만 건, 일일 신규 검색어 비율 18%
- 라벨이 있는 분류 학습 데이터 8,000건
- 응답 시간 목표: 요청 1건당 50ms 이하
- 운영팀은 예측 근거의 단어 설명을 원함
```

- OOV 비율, 평균 문서 길이, 클래스 불균형, 추론 시간 외에 최소 두 개의 확인 지표를 추가하세요.
- TF-IDF, FastText, Transformer 중 기준선과 비교 후보를 정하고 근거를 쓰세요.
- train/validation/test 분리와 최종 평가 지표를 제안하세요.
- 높은 학습 정확도만으로 모델을 채택하면 안 되는 이유를 작성하세요.

## 문제 25. 종합 미니 프로젝트: 문서 유사 검색

문제 1의 `documents`에 아래 질의를 더해, TF-IDF 기반 상위 2개 문서 검색기를 만드세요.

```python
query = "my dog loves walking"
```

조건:

- 코퍼스에 fit한 TF-IDF vectorizer로 질의를 transform하세요. 질의에 다시 fit하면 안 됩니다.
- 질의와 모든 문서의 코사인 유사도를 계산해 내림차순 상위 2개를 출력하세요.
- 각 결과에 문서 번호, 원문, 유사도를 함께 표시하세요.
- 질의의 `walking`이 코퍼스의 `walk`와 정확히 일치하지 않을 때 생길 수 있는 문제를 설명하세요.
- 이를 완화할 전처리 하나와 벡터화 기법 하나를 제안하고, 각각의 한계를 작성하세요.

힌트: `tfidf_vectorizer.transform([query])`, `np.argsort(scores)[::-1][:2]`

## 문제 26. 벡터화의 표현 대상 구분

다음에서 벡터 하나가 무엇을 나타내는지 답하세요: (a) TF-IDF의 한 행, (b) Word2Vec의 `king` 벡터, (c) BERT의 `[CLS]` 표현, (d) Doc2Vec 문서 벡터, (e) LDA 문서-주제 벡터.

각 항목을 **표현 대상**, **값의 성격(빈도/가중치/실수 임베딩/확률 비율)**, **단어 순서 보존 여부** 기준으로 표로 정리하세요.

## 문제 27. 고정 길이 벡터와 어휘 크기

어휘 수가 8개일 때 원-핫 벡터와 4차원 임베딩 벡터의 길이를 비교하세요. 이후 어휘 수가 80,000개로 늘어날 때 각각의 길이가 어떻게 변하는지도 작성하세요.

- 원-핫에서 `queen`과 `king`의 내적이 왜 0인지 설명하세요.
- 임베딩 차원의 축이 사람이 정한 `성별`, `왕족` 같은 이름표가 아닌 이유를 작성하세요.
- 임베딩 행렬 `(vocab_size, embedding_dim)`에서 행과 열이 각각 무엇인지 설명하세요.

## 문제 28. `CountVectorizer`의 기본 토큰화 확인

다음 문서를 기본 `CountVectorizer`와 `token_pattern=r"(?u)\b\w+\b"` 설정으로 각각 변환하세요.

```python
token_docs = ["I am A.I.!", "AI is 2026-ready."]
```

- 두 설정의 어휘 목록을 비교하세요.
- 기본 설정에서 한 글자 토큰 `I`, `A`가 어떻게 처리되는지 확인하세요.
- 소문자화, 구두점 처리, 토큰 길이 규칙이 분석 결과에 미치는 영향을 주석으로 작성하세요.
- 실제 서비스 데이터에서 tokenizer 규칙을 명시적으로 검증해야 하는 이유를 한 문장으로 쓰세요.

## 문제 29. TF-IDF 직접 계산을 한 단계씩 검증하기

문제 1의 `documents`에서 `cat`, `dog`, `my`에 대해 다음을 직접 계산하세요.

- 전체 문서 수 `N`과 각 단어의 문서 빈도 `df`를 구하세요.
- `log((1 + N) / (1 + df)) + 1`로 smoothed IDF를 계산하세요.
- `doc2`의 정규화 전 TF-IDF와 L2 정규화 후 값을 계산하세요.
- 문제 3의 `TfidfVectorizer` 출력과 `np.allclose()`로 비교하세요.
- TF가 문서 내 빈도이고 DF가 코퍼스에서 등장한 문서 수인 이유를 예시와 함께 주석으로 작성하세요.

## 문제 30. BoW·TF-IDF 오류 분석

아래 문장 쌍에서 BoW/TF-IDF 유사도가 실제 의미와 어긋날 수 있는 이유를 각각 작성하고, 완화 방법을 하나씩 제안하세요.

1. `I like this movie` / `I do not like this movie`
2. `car repair shop` / `automobile maintenance center`
3. `bank loan` / `river bank`

각 완화 방법이 완전한 해결책은 아닌 이유도 한 문장씩 작성하세요.

## 문제 31. Word2Vec 입력 코퍼스 점검

다음 세 입력 중 `gensim.Word2Vec(sentences=...)`에 적절한 것을 고르고 나머지가 부적절한 이유를 작성하세요.

```python
a = ["king is royal", "queen is royal"]
b = [["king", "is", "royal"], ["queen", "is", "royal"]]
c = ["king", "is", "royal", "queen", "is", "royal"]
```

- 바깥 리스트와 안쪽 리스트가 각각 무엇을 나타내야 하는지 설명하세요.
- 문장 경계가 사라질 때 문맥 창 학습에 어떤 문제가 생기는지 작성하세요.
- 작은 교육용 코퍼스를 반복해 학습하는 장점과 한계를 각각 적으세요.

## 문제 32. Word2Vec 하이퍼파라미터 읽기

문제 5의 `vector_size`, `window`, `min_count`, `sg`, `negative`, `epochs`, `workers`, `seed`를 각각 한 줄로 설명하세요.

추가로 다음 상황에서 어떤 값을 조정할지와 예상 효과를 작성하세요.

- 희귀 단어가 너무 많이 OOV가 된다.
- 문맥 범위를 더 넓게 보고 싶다.
- 실행 결과를 수업에서 최대한 재현하고 싶다.
- 모델이 작고 학습이 부족해 보인다.

## 문제 33. 한국어 Word2Vec 전처리 설계

다음 한국어 리뷰를 Word2Vec 입력용 문장별 토큰 목록으로 만드는 코드를 작성하세요.

```python
reviews = ["영화가 정말 재밌었다!", "배우 연기는 좋지만 이야기가 지루해요."]
korean_stopwords = {"이", "가", "은", "는", "을", "를", "하지만"}
```

- 정규표현식으로 한글·공백 이외 문자를 처리하고 연속 공백을 정리하세요.
- `Okt().morphs(text, stem=True)`를 사용해 형태소를 추출하세요.
- 불용어와 한 글자 토큰을 제외하세요.
- 공백 분리보다 형태소 분석을 고려하는 이유와, 불용어를 무조건 제거하면 안 되는 경우를 주석으로 작성하세요.

## 문제 34. 한국어 Word2Vec 결과 해석

문제 33에서 얻은 `korean_sentences`로 `vector_size=50`, `window=5`, `min_count=3`, `sg=1` 모델을 학습한다고 가정합니다.

- `min_count=3`이 어휘에 미치는 영향을 설명하세요.
- 관심 단어 하나를 골라 등록 여부를 확인하고 `most_similar(topn=10)`을 출력하는 코드를 작성하세요.
- 비슷한 단어 목록에 예상 밖의 단어가 나왔을 때 점검할 세 가지를 작성하세요.
- 긍정·부정 레이블이 있어도 Word2Vec 학습 자체는 왜 비지도 신호를 사용하는지 설명하세요.

## 문제 35. 사전학습 Word2Vec 사용 전 점검

`word2vec-google-news-300` 같은 사전학습 벡터를 사용할 때 아래를 조사·점검하는 순서로 정리하세요.

- 어휘 수, 벡터 차원, 파일 크기와 메모리 요구량
- 언어와 도메인이 현재 데이터와 맞는지
- 라이선스와 배포 조건
- OOV 비율
- `KeyedVectors.load_word2vec_format(..., binary=True, limit=...)`의 `binary`, `limit` 의미

마지막으로 `limit`이 메모리 사용량은 줄일 수 있어도 원본 다운로드 크기 자체를 줄이지는 않는 이유를 작성하세요.

## 문제 36. `nn.Embedding`과 원-핫 선형계층의 동치

임베딩 가중치 `W`가 `(vocab_size, embedding_dim)`일 때, ID 2의 원-핫 벡터 `e_2`와 `W`의 행 조회가 같은 결과임을 확인하세요.

- `F.one_hot()`으로 ID 2의 원-핫 벡터를 만드세요.
- `one_hot.float() @ embedding.weight`와 `embedding(torch.tensor(2))`를 비교하세요.
- 두 값이 일치하는지 출력하세요.
- `nn.Embedding`이 큰 어휘에서 원-핫 행렬을 매번 만들지 않아도 되는 이유를 설명하세요.

힌트: `import torch.nn.functional as F`

## 문제 37. `MeanEmbeddingClassifier` 직접 작성

문제 6의 ID 배치를 받아 `임베딩 조회 → PAD 제외 평균 → 선형 분류`를 수행하는 `nn.Module` 클래스를 작성하세요.

- `__init__`에서 embedding과 classifier를 정의하세요.
- `forward()`에서 마스크를 만들어 실제 토큰 수로 평균을 내세요.
- 입력 `(batch, length)`에서 로짓 `(batch, class_count)`가 나오는지 확인하세요.
- PAD만 있는 문장이 들어와도 0으로 나누지 않도록 `clamp(min=1)`을 사용하세요.
- 평균 풀링이 토큰 순서를 잃는 예를 하나 작성하세요.

## 문제 38. 역전파로 임베딩이 갱신되는지 확인

문제 37의 분류기를 한 번 학습시키기 전후로 `good` 토큰 행의 임베딩 값을 비교하세요.

- 학습 전 `classifier.embedding.weight[word_to_id['good']]`을 복사하세요.
- `zero_grad() → forward() → CrossEntropyLoss → backward() → step()`을 한 번 실행하세요.
- 학습 후 같은 행과 `torch.allclose()` 결과를 출력하세요.
- `<PAD>` 행도 비교하고, `padding_idx`가 있는 경우 왜 다른 결과가 나와야 하는지 설명하세요.
- `CrossEntropyLoss` 앞에 softmax를 추가하면 안 되는 이유를 주석으로 작성하세요.

## 문제 39. 학습과 평가 모드 분리

문제 37의 모델을 30 epoch 학습한 후 평가 코드를 작성하세요.

- 학습 단계에서 `model.train()`을 호출하세요.
- 평가 단계에서 `model.eval()`과 `torch.no_grad()`를 사용하세요.
- 로짓 shape, 예측 ID, 정답 ID, 정확도를 출력하세요.
- `argmax(dim=1)`이 어떤 축에서 어떤 값을 고르는지 설명하세요.
- 학습 문장 정확도가 높아도 일반화 성능이라고 볼 수 없는 이유를 작성하세요.

## 문제 40. 사전학습 임베딩의 freeze 선택

문제 8의 `frozen_embedding`, `trainable_embedding`을 optimizer에 각각 넣고 한 번의 역전파 후 `movie` 행 변화를 비교하세요.

- 두 weight의 `requires_grad`를 확인하세요.
- 두 경우에 `movie` 행이 변했는지 출력하세요.
- 작은 데이터·도메인 일치 상황과 충분한 데이터·도메인 차이 상황에서 freeze 선택을 각각 제안하세요.
- PAD 행을 직접 0으로 초기화해야 할 수 있는 이유를 설명하세요.

## 문제 41. FastText 경계 기호와 여러 길이 n-gram

`character_ngrams()` 함수를 사용해 `apple`의 3-gram, 4-gram, 5-gram, 6-gram을 각각 출력하세요.

- 시작·끝 경계 기호 `<`, `>`를 붙인 결과에서만 나타나는 조각을 찾으세요.
- `apple`과 `apples`가 3~6-gram에서 공유하는 조각을 길이별로 출력하세요.
- 경계 기호가 없을 때 `apple`의 처음·끝 부분과 단어 내부를 구별하기 어려운 이유를 설명하세요.
- `min_n`, `max_n`을 늘렸을 때 장점과 비용을 각각 작성하세요.

## 문제 42. 공정한 Word2Vec·FastText 비교 실험

동일한 `training_sentences`로 두 모델을 비교하려면 무엇을 같게 유지해야 하는지 작성하고, 코드에서 `common_settings` 딕셔너리를 만드세요.

- `vector_size`, `window`, `min_count`, `sg`, `negative`, `epochs`, `workers`, `seed`를 공통 설정으로 두세요.
- 두 모델의 등록 어휘 행렬 shape과 등록 어휘 수를 출력하세요.
- 행렬 shape이 같아도 FastText가 문자 n-gram 정보를 추가로 학습한다는 사실을 shape만으로 알 수 없는 이유를 설명하세요.
- 두 모델 비교에서 코퍼스·난수·학습 횟수까지 통제해야 하는 이유를 작성하세요.

## 문제 43. 등록 어휘 여부와 OOV 벡터 생성을 구분하기

문제 9의 Word2Vec·FastText 모델에서 `apple`, `apples`, `teacher`, `qzxw`를 비교하세요.

- 각 단어가 `key_to_index`에 등록됐는지 표로 출력하세요.
- Word2Vec과 FastText 각각에서 `get_vector()` 성공 여부와 벡터 shape을 확인하세요. 예외는 `try-except`로 처리합니다.
- FastText에서 ‘등록 어휘가 아님’과 ‘벡터를 만들 수 없음’이 다른 판단인 이유를 설명하세요.
- 무작위 문자열 `qzxw`에도 FastText 벡터가 나올 수 있을 때, 이것을 신뢰할 수 있는 의미 표현이라고 바로 판단하면 안 되는 이유를 작성하세요.

## 문제 44. FastText 유사도 결과 비판적으로 읽기

FastText에서 다음 유사도를 계산하고 내림차순으로 정렬하세요.

```python
pairs = [("apple", "apples"), ("teach", "teacher"), ("apple", "teacher"), ("cat", "cut")]
```

- 철자 공유와 문맥 공유가 각각 유사도에 미칠 수 있는 영향을 설명하세요.
- 문자 n-gram을 bucket에 해싱할 때 생길 수 있는 충돌 가능성을 한 문장으로 작성하세요.
- 오탈자가 많은 검색어와 철자가 비슷하지만 의미가 다른 전문 용어 중 FastText가 더 유리할 가능성이 큰 경우를 구분해 설명하세요.
- 실제 채택 전 어떤 다운스트림 평가를 해야 하는지 제안하세요.

## 문제 45. 최종 종합: 방법 선택 보고서

아래 세 프로젝트에 대해 **첫 번째 기준선**, **비교 후보**, **평가 지표**, **가장 큰 위험**을 표로 작성하세요.

1. 근거 단어를 보여줘야 하는 사내 문의 분류
2. 신조어·오탈자가 많은 이커머스 검색어 매칭
3. 문맥에 따라 다른 뜻을 갖는 금융 뉴스 문장 분류

각 프로젝트에서 BoW/TF-IDF, Word2Vec, FastText, 사전학습 임베딩, BERT 계열 중 선택 근거를 쓰세요. 마지막으로 ‘가장 복잡한 모델을 바로 쓰지 않는’ 실무적 이유를 데이터·비용·설명 가능성 관점에서 정리하세요.

# PyTorch `nn.Embedding` 단계별 추가 실습

문제 46~60은 앞 문제 6~8, 36~40을 보완하는 단계형 연습입니다. 위에서부터 순서대로 풀면 문자열을 임베딩 기반 분류 모델의 입력으로 바꾸고 학습·평가할 수 있습니다.

## 문제 46. PyTorch 텐서의 shape와 dtype

다음 두 텐서를 만들고 값·shape·dtype을 출력하세요.

```python
token_ids = [[2, 5, 0], [4, 3, 1]]
scores = [[0.2, 1.3], [2.1, -0.4]]
```

- `token_ids`는 `torch.long`, `scores`는 `torch.float32`로 만드세요.
- `nn.Embedding` 입력 ID에 정수형 `long`이 필요한 이유를 주석으로 작성하세요.
- 분류 로짓과 임베딩 출력이 실수형인 이유를 작성하세요.
- `(2, 3)` shape이 배치·시퀀스 관점에서 무엇을 뜻하는지 설명하세요.

## 문제 47. 단어 사전과 OOV 인코더 만들기

다음 학습 문장으로 `word_to_id`, `id_to_word`, `encode(tokens)` 함수를 만드세요.

```python
train_sentences = [["i", "love", "nlp"], ["nlp", "is", "fun"]]
test_sentence = ["i", "enjoy", "nlp"]
```

- `<PAD>`는 0, `<OOV>`는 1로 고정하세요.
- 나머지 단어에는 2부터 ID를 부여하세요.
- `test_sentence`를 인코딩하고 `enjoy`가 OOV ID가 되는지 확인하세요.
- 임베딩의 `num_embeddings`가 왜 `max ID + 1` 이상이어야 하는지 설명하세요.

## 문제 48. 길이가 다른 문장 패딩하기

문제 47의 학습 문장과 아래 문장을 ID로 바꾼 뒤 최대 길이 5로 맞추세요.

```python
extra_sentence = ["i", "love", "fun", "nlp"]
```

- 뒤쪽 패딩(post-padding)을 직접 구현해 `(문장 수, 5)` 텐서를 만드세요.
- 앞쪽 패딩(pre-padding) 결과도 만들어 첫 문장의 차이를 비교하세요.
- 길이 5를 넘는 입력은 뒤쪽 절단(post-truncating)하세요.
- RNN 계열에서 패딩 위치가 달라질 수 있는 이유와, 평균 풀링에서는 무엇을 더 주의해야 하는지 작성하세요.

## 문제 49. 눈으로 확인하는 임베딩 행 조회

랜덤 초기값 대신 알아보기 쉬운 가중치로 `nn.Embedding(6, 3)`를 만드세요.

- `torch.arange(18).reshape(6, 3)`을 weight로 복사하세요.
- 입력 ID `[2, 5, 1]`의 임베딩 결과를 출력하세요.
- 입력의 각 ID와 선택된 weight 행을 반복문으로 함께 출력하세요.
- ID 값 5가 ‘크기가 큰 단어’라는 의미가 아닌 이유를 주석으로 작성하세요.

## 문제 50. 배치 입력에서 임베딩 출력 shape 예측

아래 입력을 `nn.Embedding(num_embeddings=10, embedding_dim=4)`에 넣기 전, 출력 shape을 먼저 예상하고 실행으로 확인하세요.

```python
one_sentence = torch.tensor([2, 5, 1])
batch_sentences = torch.tensor([[2, 5, 1], [4, 0, 0]])
```

- 두 입력과 출력의 shape을 각각 출력하세요.
- 왜 입력의 마지막 축이 사라지지 않고 embedding 차원이 뒤에 추가되는지 설명하세요.
- 배치 차원, 토큰 위치 차원, 임베딩 차원을 각각 표시하세요.

## 문제 51. `padding_idx`의 역할 확인

`nn.Embedding(6, 3, padding_idx=0)`과 `nn.Embedding(6, 3)`을 각각 만들고 PAD 행을 비교하세요.

- 두 계층의 0번 행을 출력하세요.
- 입력 `[0, 2, 0, 5]`를 넣어 PAD 위치의 출력 벡터를 확인하세요.
- `padding_idx=0`이 패딩 위치를 텐서에서 삭제하는 기능이 아닌 이유를 설명하세요.
- 사전학습 행렬로 `from_pretrained()`를 쓸 때 PAD 행을 직접 0으로 설정할 수 있는 이유를 작성하세요.

## 문제 52. 마스크 shape와 broadcasting

문제 50의 `batch_sentences`와 임베딩 출력 `embedded`를 사용해 PAD 마스크를 만드세요.

- `token_mask = batch_sentences.ne(0)`의 값과 shape을 출력하세요.
- `expanded_mask = token_mask.unsqueeze(-1)`의 shape을 출력하세요.
- `embedded * expanded_mask`가 가능한 이유를 broadcasting 관점에서 설명하세요.
- `unsqueeze(-1)`를 빼면 어떤 차원 문제가 생길 수 있는지 확인하세요.

## 문제 53. 패딩을 제외한 평균 풀링

문제 52의 `embedded`, `expanded_mask`로 문장 벡터를 만드세요.

- PAD를 포함한 단순 평균 `embedded.mean(dim=1)`을 구하세요.
- 마스크 합과 실제 토큰 수를 이용한 평균을 구하세요.
- 두 번째 문장처럼 PAD가 많은 경우 두 결과를 비교하세요.
- `sum(dim=1)`, `sum(dim=1).clamp(min=1)`에서 각 `dim=1`이 의미하는 축을 설명하세요.
- `<PAD>` 벡터가 0이어도 단순 평균이 틀릴 수 있는 이유를 작성하세요.

## 문제 54. 평균 풀링 분류기의 forward 추적

문제 37의 `MeanEmbeddingClassifier`를 사용하거나 다시 구현해, 다음 단계별 shape을 출력하세요.

1. 입력 ID
2. token embedding
3. PAD 제외 문장 벡터
4. 두 클래스의 logits

- 각 단계의 shape과 값 하나를 출력하세요.
- 로짓 두 값이 바로 확률이 아닌 이유를 설명하세요.
- `softmax(logits, dim=1)` 결과의 각 행 합이 1인지 확인하세요.
- 예측 단계와 `CrossEntropyLoss` 학습 단계에서 softmax를 다르게 다루는 이유를 작성하세요.

## 문제 55. 분류 정답(label) 형식 점검

이진 감성 분류에서 아래 세 label 표현을 만들고 `CrossEntropyLoss`에 알맞은 것을 고르세요.

```python
class_ids = [1, 0, 1]
one_hot_labels = [[0, 1], [1, 0], [0, 1]]
```

- `(batch, 2)` logits와 함께 사용할 target의 dtype과 shape을 출력하세요.
- class ID target과 one-hot target의 차이를 설명하세요.
- 클래스 번호 0·1이 긍정/부정의 크기 순서를 뜻하지 않는 이유를 주석으로 작성하세요.
- label shape 오류가 나면 가장 먼저 무엇을 확인할지 적으세요.

## 문제 56. 한 번의 학습 step을 함수로 만들기

`train_step(model, inputs, targets, optimizer, criterion)` 함수를 작성하세요.

- 함수 안에서 `model.train()`을 호출하세요.
- 기울기 초기화, 로짓 계산, 손실 계산, 역전파, optimizer step을 순서대로 구현하세요.
- 반환값으로 loss의 Python float 값을 돌려주세요.
- 함수 밖에서 10회 호출하고 loss 목록을 출력하세요.
- `optimizer.zero_grad()`를 생략하면 생기는 문제를 주석으로 작성하세요.

## 문제 57. 배치 학습과 DataLoader

문제 48에서 만든 padded ID 텐서와 label 텐서를 `TensorDataset`, `DataLoader(batch_size=2, shuffle=True)`로 묶으세요.

- 한 epoch에서 나오는 각 mini-batch 입력·정답 shape을 출력하세요.
- 문제 56의 `train_step()`으로 mini-batch마다 학습하세요.
- epoch 평균 loss를 계산해 5 epoch 동안 출력하세요.
- `shuffle=True`의 목적과, 검증·테스트 DataLoader에서 보통 `shuffle=False`를 쓰는 이유를 작성하세요.

힌트: `from torch.utils.data import TensorDataset, DataLoader`

## 문제 58. 예측 확률과 평가 함수

`evaluate(model, data_loader)` 함수를 작성해 평균 loss, accuracy, 예측 ID를 반환하세요.

- 함수 안에서 `model.eval()`과 `torch.no_grad()`를 사용하세요.
- logits에서 `argmax(dim=1)`으로 예측 class ID를 얻으세요.
- `softmax(dim=1)`로 긍정 클래스 확률도 출력하세요.
- accuracy를 `맞은 개수 / 전체 개수`로 직접 계산하세요.
- 이진 분류에서 accuracy만으로 부족할 수 있는 데이터 상황을 하나 작성하세요.

## 문제 59. 사전학습 임베딩 행렬 정렬 실습

아래 사전학습 벡터 사전과 문제 47의 `word_to_id`로 임베딩 행렬을 만드세요.

```python
pretrained_vectors = {
    "i": [0.1, 0.2, 0.3, 0.4],
    "love": [0.7, 0.2, 0.1, 0.5],
    "nlp": [0.3, 0.8, 0.4, 0.2],
}
```

- `(len(word_to_id), 4)` 크기의 행렬을 만들고 PAD 행은 0으로 두세요.
- 사전에 없는 `fun`, `is`, `<OOV>`의 초기화 규칙을 정하고 주석으로 설명하세요.
- `from_pretrained(..., freeze=True/False)` 두 계층을 만드세요.
- `id_to_word`를 사용해 각 행이 올바른 단어 벡터인지 검증하는 코드를 작성하세요.

## 문제 60. 종합: 새 문장 분류 함수 만들기

문제 47~59의 결과를 연결해 문자열 문장 하나를 입력받는 `predict_sentiment(text)` 함수를 작성하세요.

- 소문자화와 공백 분리로 토큰화하세요.
- `word_to_id`로 변환하되 처음 보는 단어는 `<OOV>`로 바꾸세요.
- 학습 때와 같은 최대 길이·post-padding 규칙을 적용하세요.
- 모델 평가 모드와 `torch.no_grad()`에서 logits, 예측 class, softmax 확률을 반환하세요.
- `"i love nlp"`, `"i enjoy nlp"`, 빈 문자열을 입력해 동작을 확인하세요.
- 이 작은 예제 모델의 예측을 실제 감성 판단으로 신뢰할 수 없는 이유와, 다음 개선 단계 하나를 작성하세요.